# Embedding geometry

Fit PCA on training embeddings, then project held-out evaluation embeddings.
The top row shows training explained variance; the bottom row shows evaluation geometry.
Download the analysis data as described in the root README, or run `scripts/export_embeddings.py`.
Select a method and edit the component/camera settings below.


In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "planning/train.py").is_file())
PLANNING = ROOT / "planning"
sys.path.insert(0, str(PLANNING))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from plot_style import apply_matplotlib_style

apply_matplotlib_style()

from sklearn.decomposition import PCA
from analysis.probes import load_split, physical_targets

EXPORT_ROOT = PLANNING / "outputs/embeddings"
METHOD = "smwm"  # smwm, sigreg, forward_only
VIEWS = {  # zero-based component order, elevation, azimuth
    "tworoom": ((0, 1, 2), 22, -60),
    "reacher": ((0, 1, 2), 22, -60),
    "pusht": ((0, 1), 0, 0),
    "ogbcube": ((0, 2, 1), 22, -60),
}


In [ ]:
fig = plt.figure(figsize=(9, 4.2), layout="constrained")
loaded = 0
for column, (env, label) in enumerate(zip(VIEWS, ["TwoRoom", "Reacher", "Push-T", "OGBench-Cube"])):
    directory = EXPORT_ROOT / env / METHOD
    spectrum = fig.add_subplot(2, 4, column + 1)
    pcs, elevation, azimuth = VIEWS[env]
    geometry = fig.add_subplot(2, 4, column + 5, projection="3d" if len(pcs) == 3 else None)
    if not (directory / "train_embeddings.pt").is_file() or not (directory / "eval_embeddings.pt").is_file():
        spectrum.set(title=label)
        spectrum.text(0.5, 0.5, "Export embeddings first", ha="center", transform=spectrum.transAxes)
        spectrum.set_axis_off()
        geometry.set_axis_off()
        continue
    z_train, _ = load_split(directory, "train")
    z_eval, arrays = load_split(directory, "eval")
    pca = PCA().fit(z_train)
    projected = pca.transform(z_eval)
    spectrum.bar(np.arange(1, 6), pca.explained_variance_ratio_[:5] * 100, color="#2066a8")
    spectrum.set(title=label, xlabel="Principal component", ylabel="Variance (%)")
    spectrum.spines[["top", "right"]].set_visible(False)
    quantity, values = next(iter(physical_targets(env, arrays).items()))
    color = np.arctan2(values[:, 0], values[:, 1]) if env == "reacher" else values[:, 0]
    points = geometry.scatter(*[projected[:, pc] for pc in pcs], c=color, cmap="viridis",
                              s=2, alpha=0.65, linewidths=0, rasterized=True)
    geometry.set(xlabel=f"PC{pcs[0]+1}", ylabel=f"PC{pcs[1]+1}")
    if len(pcs) == 3:
        geometry.set_zlabel(f"PC{pcs[2]+1}", labelpad=8)
        geometry.view_init(elev=elevation, azim=azimuth)
    fig.colorbar(points, ax=geometry, shrink=0.65, label=quantity)
    loaded += 1
if loaded:
    output = PLANNING / "outputs/analysis"
    output.mkdir(parents=True, exist_ok=True)
    fig.savefig(output / f"embeddings_{METHOD}.pdf", bbox_inches="tight")
plt.show()
